# DC Crash Data Exploration

This notebook explores the raw Open Data DC crash dataset before building the cleaning and hotspot-screening pipeline.

The goal is not to fully clean the data yet. The goal is to understand:
1. What fields are available
2. Which fields matter for location quality
3. Which fields matter for crash severity
4. What missingness or data-quality problems exist
5. What cleaning decisions should be made in the pipeline

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Crashes_in_DC.csv", low_memory=False)

df.shape

(347809, 66)

## Initial Size

The dataset contains about 348,000 crash records and 66 columns. This is large enough to require systematic cleaning, but small enough to explore locally in Python.

In [2]:
df.head()

,X,Y,CRIMEID,CCN,REPORTDATE,ROUTEID,MEASURE,OFFSET,STREETSEGID,ROADWAYSEGID,...,MAR_ID,BLOCKKEY,SUBBLOCKKEY,CORRIDORID,NEARESTINTKEY,MAJORINJURIESOTHER,MINORINJURIESOTHER,UNKNOWNINJURIESOTHER,FATALOTHER,OBJECTID
0,-8.574282e+06,4.709920e+06,23586905,11017754,2011/02/09 21:30:00+00,11036722,1547.46,33.57,-9.0,25816.0,...,285452,e089662cd2ae74b282206b3c3212d287,e089662cd2ae74b282206b3c3212d287,Blockkey Not Found on Corridor,5c930306b9518150c67e00540289058e,NaN,NaN,NaN,NaN,470575118
1,-8.581573e+06,4.715519e+06,23587032,11018463,2011/02/11 10:30:00+00,11043552,1752.86,0.01,7576.0,5766.0,...,808848,176670a090508d0a8722966a591bfa75,176670a090508d0a8722966a591bfa75,11043552_3,59889a2d374316ab367aa592088c1593,NaN,NaN,NaN,NaN,470575119
2,-8.575957e+06,4.708738e+06,23587090,11018425,2011/02/11 02:15:00+00,11068382,2486.28,0.07,641.0,11005.0,...,225843,15c7a27804c91fd21054e9ead0210b8f,d770b155f9ced6eb651eae4605b618a3,11068382_2,d0818414493dcb3e27e9ed0f6b12382f,NaN,NaN,NaN,NaN,470575120
3,-8.574416e+06,4.713351e+06,23587360,11018342,2011/02/10 05:00:00+00,11088012,1060.34,0.09,9686.0,9984.0,...,811305,de51a85287b764641a0d3ff5fb986a4c,de51a85287b764641a0d3ff5fb986a4c,11088012_1,0a87a6089c9dd10746382b0be0a2f186,NaN,NaN,NaN,NaN,470575121
4,-8.569765e+06,4.708381e+06,23587635,11018068,2011/02/10 12:50:00+00,12086642,705.97,0.00,1075.0,13417.0,...,903559,1ab7e3e8d763d4f65595212998c38643,0ac56a1636d1a2a40bcdf8b0104c3e33,12086642_1,ef86b66671597bb4d220c2777c75b8e1,NaN,NaN,NaN,NaN,470575122


In [3]:
df.columns.tolist()

['X',
 'Y',
 'CRIMEID',
 'CCN',
 'REPORTDATE',
 'ROUTEID',
 'MEASURE',
 'OFFSET',
 'STREETSEGID',
 'ROADWAYSEGID',
 'FROMDATE',
 'TODATE',
 'ADDRESS',
 'LATITUDE',
 'LONGITUDE',
 'XCOORD',
 'YCOORD',
 'WARD',
 'EVENTID',
 'MAR_ADDRESS',
 'MAR_SCORE',
 'MAJORINJURIES_BICYCLIST',
 'MINORINJURIES_BICYCLIST',
 'UNKNOWNINJURIES_BICYCLIST',
 'FATAL_BICYCLIST',
 'MAJORINJURIES_DRIVER',
 'MINORINJURIES_DRIVER',
 'UNKNOWNINJURIES_DRIVER',
 'FATAL_DRIVER',
 'MAJORINJURIES_PEDESTRIAN',
 'MINORINJURIES_PEDESTRIAN',
 'UNKNOWNINJURIES_PEDESTRIAN',
 'FATAL_PEDESTRIAN',
 'TOTAL_VEHICLES',
 'TOTAL_BICYCLES',
 'TOTAL_PEDESTRIANS',
 'PEDESTRIANSIMPAIRED',
 'BICYCLISTSIMPAIRED',
 'DRIVERSIMPAIRED',
 'TOTAL_TAXIS',
 'TOTAL_GOVERNMENT',
 'SPEEDING_INVOLVED',
 'NEARESTINTROUTEID',
 'NEARESTINTSTREETNAME',
 'OFFINTERSECTION',
 'INTAPPROACHDIRECTION',
 'LOCATIONERROR',
 'LASTUPDATEDATE',
 'MPDLATITUDE',
 'MPDLONGITUDE',
 'MPDGEOX',
 'MPDGEOY',
 'FATALPASSENGER',
 'MAJORINJURIESPASSENGER',
 'MINORINJURIESPASSEN

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 347809 entries, 0 to 347808
Data columns (total 66 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   X                           347809 non-null  float64
 1   Y                           347809 non-null  float64
 2   CRIMEID                     347809 non-null  int64  
 3   CCN                         347809 non-null  object 
 4   REPORTDATE                  346427 non-null  object 
 5   ROUTEID                     347809 non-null  object 
 6   MEASURE                     347809 non-null  float64
 7   OFFSET                      347809 non-null  float64
 8   STREETSEGID                 215903 non-null  float64
 9   ROADWAYSEGID                215903 non-null  float64
 10  FROMDATE                    347736 non-null  object 
 11  TODATE                      0 non-null       float64
 12  ADDRESS                     347727 non-null  object 
 13  LATITUDE      

## Initial Dataset Patterns

The dataset has one row per crash record and includes fields for location, date, severity, road-network matching, and crash context.

Important columns include latitude/longitude, address, ward, location error, MAR score, roadway/intersection identifiers, and injury/fatality counts.

In [5]:
missing_table = (
    df.isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .reset_index()
)

missing_table.columns = ["column", "missing_percent"]

missing_table.head(30)

,column,missing_percent
0,TODATE,100.00
1,LOCATIONERROR,85.56
2,FATALOTHER,72.90
3,UNKNOWNINJURIESOTHER,72.90
4,MINORINJURIESOTHER,72.90
5,MAJORINJURIESOTHER,72.90
6,LASTUPDATEDATE,63.91
7,MPDGEOX,61.94
8,MPDGEOY,61.94
9,STREETSEGID,37.92


## Executive Summary / Reframing After Exploration
(Summarizes the main finding from the exploration below)

My original hypothesis was that DC would have many unmappable crash records, similar to the Madison case study discussed by Citian. After exploring the data, I found that almost all DC crash records have valid-looking latitude and longitude values. This means the main issue is not missing coordinates.

Instead, the stronger problem is that many crashes are technically mapped but still have road-network matching issues. In the 2024–2026 subset, about 20.6% of records are classified as medium location quality because they have errors such as `FAR_FROM_CENTERLINE`, `INTERSECTING_ROUTE_ERROR`, or `BLOCKKEY_ERROR`.

This matters because these medium-quality records have nearly the same average severity as high-quality records. Excluding them would remove important safety events, but including them without caution could distort corridor, road-segment, or intersection-level screening.

Therefore DC's problem isn't missing crashes — it's crashes pinned to the wrong spot on the map.

## Missingness Notes

The missingness table shows that missing values mean different things depending on the column.

`TODATE` is 100% missing, so it is not useful for this project.

`LOCATIONERROR` is missing for 85.56% of records. Probably means that no location error was reported. That means roughly 14.44% of crashes have some kind of location error.

Several “other” injury columns are missing for 72.90% of records. These may reflect categories that were not always recorded or only apply in certain cases. Will not automatically treat these as zero without checking the data dictionary or comparing them with other injury fields.

`STREETSEGID` and `ROADWAYSEGID` are each missing for 37.92% of records. This is important because these fields connect crash records to the road network. A crash may have latitude and longitude but still be hard to use for network screening if it cannot be matched to a roadway segment.

`MPDLATITUDE` and `MPDLONGITUDE` are missing for 18.92% of records. These appear to be police-reported coordinates, so they may be useful as a backup/comparison point, but cannot rely on these alone.

Most core fields are nearly complete: `REPORTDATE`, `ADDRESS`, `FROMDATE`, `NEARESTINTROUTEID`, and `NEARESTINTKEY` have very low missingness. This suggests the dataset has enough information for a first-pass cleaning and hotspot-screening pipeline.

Overall: data is usable.

In [6]:
df["LOCATIONERROR"].value_counts(dropna=False).head(20)

LOCATIONERROR
NaN                                                                                                     297590
Crash point too far from centerline (>50m)                                                               16871
Intersecting Route ERROR.  0 Intersecting RouteID Not Found;                                              3205
8ea5a4013e07bf3928cc913a80431d7a Blockkey ERROR. 8ea5a4013e07bf3928cc913a80431d7a Blockkey ERROR.  N       589
3dda6f37c0286a68bc3077286332de14 Blockkey ERROR.  Not Found on a Corridor                                  344
d2e3b266004d70d83d8120bba7772cac Blockkey ERROR.  Not Found on a Corridor                                  303
bb08afd9df9c52b9918d9d086ceece95 Blockkey ERROR. bb08afd9df9c52b9918d9d086ceece95 Blockkey ERROR.  N       298
cc2811b57a4ff797513195227918e3ac Blockkey ERROR. cc2811b57a4ff797513195227918e3ac Blockkey ERROR.  N       278
616977ec82d8c00277a551ed2ab8f0e6 Blockkey ERROR. 616977ec82d8c00277a551ed2ab8f0e6 Blockkey ERROR. 

## Location Error Notes

`LOCATIONERROR` shows that most records have no flagged location error, but a meaningful number of records do have geospatial matching problems.

The most common flagged issue is `Crash point too far from centerline (>50m)`, with 16,871 records. This means many crashes have coordinates but may not be close enough to the road centerline to support reliable network screening.

The second major issue is `Intersecting Route ERROR`, which suggests problems matching crashes to intersections or route IDs.

Also Location Quality is not binary. Many crash records have coordinates, but some are still flagged as too far from the centerline or unable to match to a route, block, or corridor. For a CRASH-style workflow, these records are important because they may be technically geocoded but still unreliable for network-level analysis.

Therefore, for the cleaning pipeline, I will classify location errors into broader groups:
- no_error_flagged
- far_from_centerline
- intersecting_route_error
- blockkey_error
- corridor_error
- other_location_error

In [7]:
df["LOCATIONERROR"].nunique(dropna=True)

2349

In [8]:
num_location_errors = df["LOCATIONERROR"].notna().sum()
total_records = len(df)
percent_location_errors = num_location_errors / total_records * 100

num_location_errors, round(percent_location_errors, 2)

(np.int64(50219), np.float64(14.44))

Early location-quality finding: 14.44% of crash records have a flagged location error, and the raw error field contains 2,349 unique strings. Because many of these strings represent repeated versions of the same underlying error, cleaning should focus on collapsing them into useful categories rather than treating each raw string separately.

# First Cleaning Pass

Goal: Create a cleaner crash-level dataset that can support:
1. Location-quality analysis
2. Severity scoring
3. Recent crash filtering
4. Hotspot screening

In [11]:
# Make a working copy of the data
clean = df.copy()

In [12]:
# converts REPORTDATE and FROMDATE into real datetime columns.
clean["REPORTDATE"] = pd.to_datetime(clean["REPORTDATE"], errors="coerce")
clean["FROMDATE"] = pd.to_datetime(clean["FROMDATE"], errors="coerce")

In [14]:
# Filter to 2024 - 2026
clean["crash_year"] = clean["REPORTDATE"].dt.year
# Looks at seasonal/monthly patterns
clean["crash_month"] = clean["REPORTDATE"].dt.month
# Gives date without full timestamp
clean["crash_date"] = clean["REPORTDATE"].dt.date

In [16]:
# Standardize text fields
text_cols = [
    "ADDRESS",
    "MAR_ADDRESS",
    "WARD",
    "LOCATIONERROR",
    "NEARESTINTSTREETNAME",
    "INTAPPROACHDIRECTION"
]

for col in text_cols:
    if col in clean.columns:
        clean[col] = clean[col].astype("string").str.strip().str.upper()

In [17]:
# Clean Coordinates (so all coordinate-related columns are numeric)
coord_cols = ["LATITUDE", "LONGITUDE", "XCOORD", "YCOORD", "MPDLATITUDE", "MPDLONGITUDE", "MPDGEOX", "MPDGEOY"]

for col in coord_cols:
    if col in clean.columns:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")

In [18]:
# Coordinates are in the correct range to be in DC
clean["has_valid_lat_lon"] = (
    clean["LATITUDE"].between(38.7, 39.1) &
    clean["LONGITUDE"].between(-77.2, -76.8)
)

In [19]:
# Boolean to ensure LOCATIONERROR has a value
clean["has_location_error"] = clean["LOCATIONERROR"].notna()

In [25]:
# This function takes one raw LOCATIONERROR value and returns a cleaner category.
def classify_location_error(error_text):
    if pd.isna(error_text):
        return "NO_ERROR_FLAGGED"

    error_text = str(error_text).upper()

    if "CENTERLINE" in error_text:
        return "FAR_FROM_CENTERLINE"
    elif "INTERSECTING ROUTE ERROR" in error_text:
        return "INTERSECTING_ROUTE_ERROR"
    elif "BLOCKKEY ERROR" in error_text and "NOT FOUND ON A CORRIDOR" in error_text:
        return "BLOCKKEY_AND_CORRIDOR_ERROR"
    elif "BLOCKKEY ERROR" in error_text:
        return "BLOCKKEY_ERROR"
    elif "NOT FOUND ON A CORRIDOR" in error_text:
        return "CORRIDOR_ERROR"
    else:
        return "OTHER_LOCATION_ERROR"

# creates a new column by APPLYING that function to every row.
clean["location_error_type"] = clean["LOCATIONERROR"].apply(classify_location_error)

In [26]:
# Check 1
clean["location_error_type"].value_counts(dropna=False)

location_error_type
NO_ERROR_FLAGGED               297590
BLOCKKEY_ERROR                  25659
FAR_FROM_CENTERLINE             16873
INTERSECTING_ROUTE_ERROR         6533
BLOCKKEY_AND_CORRIDOR_ERROR      1104
OTHER_LOCATION_ERROR               50
Name: count, dtype: int64

## Location Error Cleaning

The raw LOCATIONERROR column contains thousands of unique strings because many errors include specific blockkey IDs. I collapsed those raw strings into broader categories that are more useful for analysis.

This preserves the important information while making the field usable for summary statistics, filtering, and dashboarding.

In [36]:
# Now we can move onto injury/fatality columns
severity_cols = [
    col for col in clean.columns
    if "FATAL" in col.upper() or "INJUR" in col.upper()
]

severity_cols

['MAJORINJURIES_BICYCLIST',
 'MINORINJURIES_BICYCLIST',
 'UNKNOWNINJURIES_BICYCLIST',
 'FATAL_BICYCLIST',
 'MAJORINJURIES_DRIVER',
 'MINORINJURIES_DRIVER',
 'UNKNOWNINJURIES_DRIVER',
 'FATAL_DRIVER',
 'MAJORINJURIES_PEDESTRIAN',
 'MINORINJURIES_PEDESTRIAN',
 'UNKNOWNINJURIES_PEDESTRIAN',
 'FATAL_PEDESTRIAN',
 'FATALPASSENGER',
 'MAJORINJURIESPASSENGER',
 'MINORINJURIESPASSENGER',
 'UNKNOWNINJURIESPASSENGER',
 'MAJORINJURIESOTHER',
 'MINORINJURIESOTHER',
 'UNKNOWNINJURIESOTHER',
 'FATALOTHER']

In [38]:
# Convert to numbers
for col in severity_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

In [42]:
# IMPORTANT CLEANING DECISION
# For a first pass, we can fill missing severity counts with zero.
for col in severity_cols:
    clean[col] = clean[col].fillna(0)

## Severity Count Cleaning Decision

For injury and fatality count columns, missing values are filled with 0 in the first cleaning pass. Because these columns represent counts of people in each severity category, a blank value most likely means that no people were recorded in that category.
For example, if `FATALBICYCLIST` is missing, I interpret that as zero bicyclist fatalities for that crash rather than an unknown number of bicyclist fatalities.

This is a reasonable first-pass assumption, but it should be revisited if the data dictionary indicates that missing values mean “unknown” rather than zero.

In [43]:
# Grouping the severity columns by type
fatal_cols = [col for col in severity_cols if "FATAL" in col.upper()]
major_injury_cols = [col for col in severity_cols if "MAJOR" in col.upper()]
minor_injury_cols = [col for col in severity_cols if "MINOR" in col.upper()]
unknown_injury_cols = [col for col in severity_cols if "UNKNOWN" in col.upper()]

In [46]:
# Check what this found
fatal_cols, major_injury_cols, minor_injury_cols, unknown_injury_cols
# It should have successfully grouped the severity columns into four clean buckets

(['FATAL_BICYCLIST',
  'FATAL_DRIVER',
  'FATAL_PEDESTRIAN',
  'FATALPASSENGER',
  'FATALOTHER'],
 ['MAJORINJURIES_BICYCLIST',
  'MAJORINJURIES_DRIVER',
  'MAJORINJURIES_PEDESTRIAN',
  'MAJORINJURIESPASSENGER',
  'MAJORINJURIESOTHER'],
 ['MINORINJURIES_BICYCLIST',
  'MINORINJURIES_DRIVER',
  'MINORINJURIES_PEDESTRIAN',
  'MINORINJURIESPASSENGER',
  'MINORINJURIESOTHER'],
 ['UNKNOWNINJURIES_BICYCLIST',
  'UNKNOWNINJURIES_DRIVER',
  'UNKNOWNINJURIES_PEDESTRIAN',
  'UNKNOWNINJURIESPASSENGER',
  'UNKNOWNINJURIESOTHER'])

In [47]:
# Now can safely run the following: For each crash, add up all fatality columns into one total fatality count. Then do the same for major injuries, minor injuries, and unknown injuries.
clean["total_fatalities"] = clean[fatal_cols].sum(axis=1)
clean["total_major_injuries"] = clean[major_injury_cols].sum(axis=1)
clean["total_minor_injuries"] = clean[minor_injury_cols].sum(axis=1)
clean["total_unknown_injuries"] = clean[unknown_injury_cols].sum(axis=1)

clean["total_injuries"] = (
    clean["total_major_injuries"] +
    clean["total_minor_injuries"] +
    clean["total_unknown_injuries"]
)

In [49]:
# Check
clean[
    [
        "total_fatalities",
        "total_major_injuries",
        "total_minor_injuries",
        "total_unknown_injuries",
        "total_injuries"
    ]
].describe()

,total_fatalities,total_major_injuries,total_minor_injuries,total_unknown_injuries,total_injuries
count,347809.000000,347809.000000,347809.000000,347809.000000,347809.000000
mean,0.002018,0.082042,0.298126,0.056083,0.436251
std,0.046455,0.394079,0.656740,0.268638,0.797292
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,1.000000
max,3.000000,51.000000,19.000000,16.000000,51.000000


## Severity Total Check

The crash-level severity totals look mostly reasonable.

1. Most crashes have zero fatalities and zero or low injury counts.
2. Average number of injuries per crash is below 1.
3. Minor injuries are more common than major injuries.
4. Fatalities are rare.

Note: the maximum injury count is high: one or more records show up to 51 major injuries or total injuries. These records may represent severe multi-person crashes. They should be inspected before relying on them in severity scoring.

In [51]:
# Inspect the highest-injury rows
clean[
    [
        "REPORTDATE",
        "ADDRESS",
        "WARD",
        "total_fatalities",
        "total_major_injuries",
        "total_minor_injuries",
        "total_unknown_injuries",
        "total_injuries",
        "LOCATIONERROR"
    ]
].sort_values("total_injuries", ascending=False).head(10)

,REPORTDATE,ADDRESS,WARD,total_fatalities,total_major_injuries,total_minor_injuries,total_unknown_injuries,total_injuries,LOCATIONERROR
84858,2010-04-04 04:00:00+00:00,100 MARYLAND AVE SW,WARD 2,0.0,51.0,0.0,0.0,51.0,<NA>
61820,2009-01-22 05:00:00+00:00,LOUISIANA AVE NE & COLUMBUS CIR NE,WARD 6,0.0,49.0,2.0,0.0,51.0,<NA>
10571,2013-01-25 05:00:00+00:00,COLUMBUS CIRCLE NW,WARD 6,0.0,44.0,0.0,0.0,44.0,<NA>
63696,2009-03-12 04:00:00+00:00,9TH ST NW & K ST NW,WARD 2,0.0,33.0,0.0,0.0,33.0,<NA>
63131,2012-05-02 04:00:00+00:00,COLUMBUS CIRCLE NE,WARD 6,0.0,33.0,0.0,0.0,33.0,<NA>
883,2011-11-17 05:50:00+00:00,SOUTHERN AVE SE AND NAYLOR RD SE,NULL,0.0,31.0,0.0,0.0,31.0,<NA>
19279,2013-07-15 04:00:00+00:00,1700 NEW YORK AVE NE,WARD 2,0.0,26.0,0.0,0.0,26.0,<NA>
95670,2010-07-03 04:00:00+00:00,3RD ST & MADISON ST NW,WARD 4,0.0,19.0,2.0,0.0,21.0,<NA>
14254,2013-04-13 04:00:00+00:00,2720 MARTIN LUTHER KING JR AVE SE,WARD 8,0.0,2.0,19.0,0.0,21.0,CRASH POINT TOO FAR FROM CENTERLINE (>50M)
22708,2013-09-24 04:00:00+00:00,3300 MINNESOTA AVE SE,WARD 7,0.0,1.0,18.0,0.0,19.0,<NA>


In [52]:
# Suspicious of the 51.0 total_injuries.
# One more check: this will show the original fatality/injury columns for the top 10 rows.
high_injury_rows = clean.sort_values("total_injuries", ascending=False).head(10)

high_injury_rows[
    ["REPORTDATE", "ADDRESS", "WARD"] + severity_cols
]

,REPORTDATE,ADDRESS,WARD,MAJORINJURIES_BICYCLIST,MINORINJURIES_BICYCLIST,UNKNOWNINJURIES_BICYCLIST,FATAL_BICYCLIST,MAJORINJURIES_DRIVER,MINORINJURIES_DRIVER,UNKNOWNINJURIES_DRIVER,...,UNKNOWNINJURIES_PEDESTRIAN,FATAL_PEDESTRIAN,FATALPASSENGER,MAJORINJURIESPASSENGER,MINORINJURIESPASSENGER,UNKNOWNINJURIESPASSENGER,MAJORINJURIESOTHER,MINORINJURIESOTHER,UNKNOWNINJURIESOTHER,FATALOTHER
84858,2010-04-04 04:00:00+00:00,100 MARYLAND AVE SW,WARD 2,0,0,0,0,0,0,0,...,0,0,0,51,0,0,0.0,0.0,0.0,0.0
61820,2009-01-22 05:00:00+00:00,LOUISIANA AVE NE & COLUMBUS CIR NE,WARD 6,0,0,0,0,1,0,0,...,0,0,0,48,2,0,0.0,0.0,0.0,0.0
10571,2013-01-25 05:00:00+00:00,COLUMBUS CIRCLE NW,WARD 6,0,0,0,0,0,0,0,...,0,0,0,44,0,0,0.0,0.0,0.0,0.0
63696,2009-03-12 04:00:00+00:00,9TH ST NW & K ST NW,WARD 2,0,0,0,0,0,0,0,...,0,0,0,33,0,0,0.0,0.0,0.0,0.0
63131,2012-05-02 04:00:00+00:00,COLUMBUS CIRCLE NE,WARD 6,0,0,0,0,0,0,0,...,0,0,0,33,0,0,0.0,0.0,0.0,0.0
883,2011-11-17 05:50:00+00:00,SOUTHERN AVE SE AND NAYLOR RD SE,NULL,0,0,0,0,0,0,0,...,0,0,0,31,0,0,0.0,0.0,0.0,0.0
19279,2013-07-15 04:00:00+00:00,1700 NEW YORK AVE NE,WARD 2,0,0,0,0,2,0,0,...,0,0,0,24,0,0,0.0,0.0,0.0,0.0
95670,2010-07-03 04:00:00+00:00,3RD ST & MADISON ST NW,WARD 4,0,0,0,0,0,2,0,...,0,0,0,19,0,0,0.0,0.0,0.0,0.0
14254,2013-04-13 04:00:00+00:00,2720 MARTIN LUTHER KING JR AVE SE,WARD 8,0,0,0,0,0,1,0,...,0,0,0,2,17,0,0.0,0.0,0.0,0.0
22708,2013-09-24 04:00:00+00:00,3300 MINNESOTA AVE SE,WARD 7,0,0,0,0,0,1,0,...,0,0,0,1,17,0,0.0,0.0,0.0,0.0


## High-Injury Record Check

The highest total injury records are mostly driven by passenger injury fields, especially `MAJORINJURIESPASSENGER`. This means the high totals are not being created by the aggregation logic; they are present in the raw data.

However, many of these extreme records are older records from around 2009–2013. These may represent real multi-passenger crashes, but they could also reflect older reporting patterns or legacy data issues. Because my project will focus on recent records from 2024–2026, these older outliers may not affect the final hotspot analysis.

Cleaning decision: keep these records in the full cleaned dataset, but inspect extreme injury values and be cautious when using severity scores across the full historical dataset.

In [54]:
# Check whether RECENT records have similar extreme injury counts
recent_check = clean[
    (clean["REPORTDATE"] >= "2024-01-01") &
    (clean["REPORTDATE"] < "2027-01-01")
].copy()

recent_check[
    [
        "REPORTDATE",
        "ADDRESS",
        "WARD",
        "total_fatalities",
        "total_major_injuries",
        "total_minor_injuries",
        "total_unknown_injuries",
        "total_injuries"
    ]
].sort_values("total_injuries", ascending=False).head(10)

,REPORTDATE,ADDRESS,WARD,total_fatalities,total_major_injuries,total_minor_injuries,total_unknown_injuries,total_injuries
347573,2026-05-05 13:35:00+00:00,"CONDON TERRACE SE & YUMA STREET SE\nWASHINGTON,",WARD 8,0.0,0.0,14.0,0.0,14.0
343378,2026-02-07 05:26:00+00:00,6925 GEORGIA AVENUE NW,WARD 4,0.0,0.0,10.0,0.0,10.0
343166,2026-02-03 14:30:00+00:00,1 49TH STREET SE,WARD 7,0.0,0.0,9.0,0.0,9.0
331095,2025-05-27 01:45:00+00:00,3433 BAKER STREET NE,WARD 7,0.0,0.0,8.0,0.0,8.0
312768,2024-06-19 22:00:00+00:00,704 EASTERN AVENUE NE,WARD 7,0.0,0.0,8.0,0.0,8.0
312582,2024-06-17 01:00:00+00:00,200 MASSACHUSETTS AVENUE NW,WARD 6,0.0,0.0,8.0,0.0,8.0
321933,2024-11-28 16:40:00+00:00,"ANACOSTIA DRIVE SE\nWASHINGTON,",WARD 8,0.0,0.0,8.0,0.0,8.0
320293,2024-10-30 20:15:00+00:00,"WAGNER STREET SE & 25TH STREET SE\nWASHINGTON,",WARD 8,0.0,0.0,8.0,0.0,8.0
309450,2024-04-27 01:20:00+00:00,1024 NORTH CAPITOL STREET NW,WARD 6,0.0,0.0,8.0,0.0,8.0
327861,2025-03-30 06:07:00+00:00,"NEW JERSEY AVENUE SE\nWASHINGTON,",WARD 6,0.0,0.0,7.0,0.0,7.0


## Recent High-Injury Check

After filtering to recent records from 2024–2026, the highest injury totals are much lower than in the full historical dataset. The largest recent records show roughly 7–14 total injuries, compared with older records that showed 30–50+ injuries.

This suggests that some of the most extreme injury values are concentrated in older records. For this prototype, I am going to focus on recent data to make the analysis more manageable and reduces the risk that legacy reporting patterns dominate the severity score.

IMPORTANT SCOPE DECISION: keep the full cleaned dataset, but use the recent 2024–2026 subset as the main working dataset for hotspot screening and dashboarding.

In [93]:
# Severity score: measures only recorded human harm
# If nobody was injured or killed, the score stays 0
clean["severity_score"] = (
    clean["total_fatalities"] * 100 +
    clean["total_major_injuries"] * 10 +
    clean["total_minor_injuries"] * 3 +
    clean["total_unknown_injuries"] * 1
)

# Crash burden score: includes property-damage-only crashes with a baseline score of 1
# This is useful when we want every crash to count, but serious crashes to count more
clean["crash_burden_score"] = clean["severity_score"].where(clean["severity_score"] > 0, 1)

In [57]:
# Check recent high-severity crashes
recent_check = clean[
    (clean["REPORTDATE"] >= "2024-01-01") &
    (clean["REPORTDATE"] < "2027-01-01")
].copy()

recent_check[
    [
        "REPORTDATE",
        "ADDRESS",
        "WARD",
        "total_fatalities",
        "total_major_injuries",
        "total_minor_injuries",
        "total_unknown_injuries",
        "total_injuries",
        "severity_score",
        "location_error_type"
    ]
].sort_values("severity_score", ascending=False).head(20)

,REPORTDATE,ADDRESS,WARD,total_fatalities,total_major_injuries,total_minor_injuries,total_unknown_injuries,total_injuries,severity_score,location_error_type
313538,2024-07-02 16:37:00+00:00,1578 41ST STREET SE,WARD 7,2.0,0.0,1.0,0.0,1.0,203.0,NO_ERROR_FLAGGED
307901,2024-03-29 10:50:00+00:00,730 KENILWORTH AVENUE NE,WARD 7,1.0,1.0,4.0,0.0,5.0,122.0,NO_ERROR_FLAGGED
316635,2024-08-28 22:41:00+00:00,3814 EASTERN AVENUE NE,WARD 5,1.0,2.0,0.0,0.0,2.0,120.0,NO_ERROR_FLAGGED
345847,2026-04-01 04:08:00+00:00,3360 SOUTHERN AVENUE SE,WARD 7,1.0,1.0,2.0,0.0,3.0,116.0,NO_ERROR_FLAGGED
314794,2024-07-28 07:30:00+00:00,5021 EAST CAPITOL STREET SE,WARD 7,1.0,1.0,2.0,0.0,3.0,116.0,NO_ERROR_FLAGGED
316873,2024-08-31 16:26:00+00:00,6316 EASTERN AVENUE NE,WARD 4,1.0,1.0,2.0,0.0,3.0,116.0,NO_ERROR_FLAGGED
329146,2025-04-23 03:30:00+00:00,1601 16TH STREET SE,WARD 8,1.0,1.0,1.0,0.0,2.0,113.0,FAR_FROM_CENTERLINE
343717,2026-02-13 19:06:00+00:00,4200 KANSAS AVENUE NW,WARD 4,1.0,1.0,0.0,0.0,1.0,110.0,NO_ERROR_FLAGGED
343364,2026-02-06 21:54:00+00:00,3072 STANTON ROAD SE,WARD 8,1.0,1.0,0.0,0.0,1.0,110.0,FAR_FROM_CENTERLINE
318404,2024-09-29 07:01:00+00:00,1701 13TH STREET NW,WARD 2,1.0,1.0,0.0,0.0,1.0,110.0,NO_ERROR_FLAGGED


## Severity Score Calculation

I created a `severity_score` to turn each crash into one number that reflects how serious it was. The goal is to avoid treating every crash the same. A crash with a fatality should count much more heavily than a crash with only property damage, and a crash with a major injury should count more than a crash with only a minor injury.

For each crash, I first added together the separate fatality and injury columns across different road-user types. For example, instead of keeping separate fatality counts for drivers, passengers, pedestrians, and bicyclists, I created one `total_fatalities` column. I did the same thing for major injuries, minor injuries, and unknown injuries.

Then I assigned each type of harm a weight:

- Each fatality = 100 points
- Each major injury = 10 points
- Each minor injury = 3 points
- Each unknown injury = 1 point

Hence, scores above 100 usually indicate fatal crashes or crashes with multiple serious injuries.

If a crash has no recorded injuries or fatalities, its `severity_score` stays 0. This makes `severity_score` a measure of recorded human harm only.

I also created a separate `crash_burden_score`, which gives property-damage-only crashes a baseline score of 1. This lets every crash count in broader burden analysis while keeping the pure severity measure focused on injuries and fatalities.

# Next: Risk Factor Flags

I create simple true/false flags for important crash types. These flags make the cleaned dataset easier to filter and analyze later.

For example, instead of checking every pedestrian injury and fatality column separately, I can create one `has_pedestrian` column that tells me whether any pedestrian was involved in the crash.

These flags will be useful for the dashboard because users can filter for:
- pedestrian-involved crashes
- bicyclist-involved crashes
- speeding-involved crashes
- fatal crashes
- major-injury crashes

In [58]:
# Checks which columns Python thinks are pedestrian-related and bicyclist-related.
pedestrian_cols = [col for col in clean.columns if "PEDESTRIAN" in col.upper()]
bicyclist_cols = [col for col in clean.columns if "BICYCLIST" in col.upper() or "BICYCLE" in col.upper()]

pedestrian_cols, bicyclist_cols

(['MAJORINJURIES_PEDESTRIAN',
  'MINORINJURIES_PEDESTRIAN',
  'UNKNOWNINJURIES_PEDESTRIAN',
  'FATAL_PEDESTRIAN',
  'TOTAL_PEDESTRIANS',
  'PEDESTRIANSIMPAIRED'],
 ['MAJORINJURIES_BICYCLIST',
  'MINORINJURIES_BICYCLIST',
  'UNKNOWNINJURIES_BICYCLIST',
  'FATAL_BICYCLIST',
  'TOTAL_BICYCLES',
  'BICYCLISTSIMPAIRED'])

In [59]:
# Now create the flags
clean["has_pedestrian"] = clean[pedestrian_cols].sum(axis=1) > 0
clean["has_bicyclist"] = clean[bicyclist_cols].sum(axis=1) > 0
clean["has_fatality"] = clean["total_fatalities"] > 0
clean["has_major_injury"] = clean["total_major_injuries"] > 0

In [60]:
# Handling the speeding variable
clean["SPEEDING_INVOLVED"] = pd.to_numeric(clean["SPEEDING_INVOLVED"], errors="coerce").fillna(0)
clean["has_speeding"] = clean["SPEEDING_INVOLVED"] > 0

In [61]:
# Now check the percentages - full historical data
risk_summary = clean[
    [
        "has_pedestrian",
        "has_bicyclist",
        "has_speeding",
        "has_fatality",
        "has_major_injury"
    ]
].mean().mul(100).round(2)

risk_summary

has_pedestrian      4.93
has_bicyclist       2.08
has_speeding        2.17
has_fatality        0.19
has_major_injury    6.53
dtype: float64

## Risk Flag Summary

The risk flags show that only a small share of crashes involve the most serious or vulnerable-road-user categories.
1. Pedestrian-involved crashes make up 4.93% of records
2. Bicyclist-involved crashes make up 2.08%
3. Speeding-involved crashes make up 2.17%
4. Fatal crashes make up 0.19%
5. Major-injury crashes make up 6.53%

These percentages are useful because they show why simple crash counts are not enough. Fatal, major-injury, pedestrian, and bicyclist crashes are relatively rare, but they are especially important for safety analysis.

Simillar to what we did for `severity_score`, these flags will allow the dashboard to filter for high-priority crash types rather than treating all crashes as the same.

In [62]:
# Focus: 2024 - 2026
recent_clean_check = clean[
    (clean["REPORTDATE"] >= "2024-01-01") &
    (clean["REPORTDATE"] < "2027-01-01")
].copy()

recent_risk_summary = recent_clean_check[
    [
        "has_pedestrian",
        "has_bicyclist",
        "has_speeding",
        "has_fatality",
        "has_major_injury"
    ]
].mean().mul(100).round(2)

recent_risk_summary

has_pedestrian      4.55
has_bicyclist       3.67
has_speeding        2.88
has_fatality        0.19
has_major_injury    1.55
dtype: float64

## Full Dataset vs. Recent Risk Summary

The full dataset and the 2024–2026 subset produce slightly different risk-flag percentages.

The biggest difference is in major-injury crashes: 6.53% of the full dataset has a major injury, compared with only 1.55% in the recent subset. This suggests that older records may have different injury reporting patterns or include extreme injury outliers that would distort the prototype if included.

For the final prototype, I will use the recent 2024–2026 subset as the main working dataset, while preserving the full cleaned dataset separately.

# Location Quality Flag

Next, I create a first-pass `location_quality` flag. The purpose is to separate crash records that are likely reliable for mapping from records that may need caution before being used for road-network or hotspot analysis.

This is important because a crash can have latitude and longitude but still have location-quality problems, such as being too far from the road centerline or failing to match to a block, corridor, or route.

For this first version:
- `HIGH` means the record has valid-looking DC coordinates and no flagged location error.
- `MEDIUM` means the record has valid-looking coordinates but some location warning, such as a LOCATIONERROR or weak/missing MAR score.
- `LOW` means the record does not have valid-looking latitude/longitude.

In [64]:
# Sanity check: has_valid_lat_lon is already set; confirm no values drifted
assert clean["has_valid_lat_lon"].dtype == bool, "has_valid_lat_lon should be bool"
clean["has_valid_lat_lon"].value_counts(normalize=True).mul(100).round(2)

In [65]:
# Check what percent of the data has valid-looking coordinates
clean["has_valid_lat_lon"].value_counts(normalize=True).mul(100).round(2)

has_valid_lat_lon
True     100.0
False      0.0
Name: proportion, dtype: float64

In [66]:
# Check 'MAR_SCORE' is numeric
clean["MAR_SCORE"] = pd.to_numeric(clean["MAR_SCORE"], errors="coerce")

In [67]:
# Location quality logic
def assign_location_quality(row):
    if not row["has_valid_lat_lon"]:
        return "LOW"
    elif row["has_location_error"]:
        return "MEDIUM"
    elif pd.isna(row["MAR_SCORE"]):
        return "MEDIUM"
    elif row["MAR_SCORE"] < 80:
        return "MEDIUM"
    else:
        return "HIGH"

In [69]:
# Apply to each row
clean["location_quality"] = clean.apply(assign_location_quality, axis=1)

In [70]:
# Get percentages
clean["location_quality"].value_counts(normalize=True).mul(100).round(2)

location_quality
HIGH      85.56
MEDIUM    14.44
LOW        0.00
Name: proportion, dtype: float64

In [71]:
# Now for 2024 - 2026
recent_clean_check = clean[
    (clean["REPORTDATE"] >= "2024-01-01") &
    (clean["REPORTDATE"] < "2027-01-01")
].copy()

recent_clean_check["location_quality"].value_counts(normalize=True).mul(100).round(2)

location_quality
HIGH      79.39
MEDIUM    20.61
Name: proportion, dtype: float64

## Location Quality Results

The location-quality flag shows that most records are usable for first-pass mapping, but a meaningful share have location-quality concerns.

In the full historical dataset, 85.56% of records are classified as `HIGH` quality and 14.44% are classified as `MEDIUM` quality. Almost no records are classified as `LOW`, because nearly all records have valid-looking latitude and longitude values within the expected DC range.

In the recent 2024–2026 subset, 79.39% of records are classified as `HIGH` quality and 20.61% are classified as `MEDIUM` quality. This means roughly 1 in 5 recent crash records has a location-quality warning under this first-pass framework.

This is important because the location problem is not mainly missing coordinates. Instead, many records have coordinates but may still have network-matching issues, such as blockkey errors, centerline-distance errors, or route/intersection errors.

These `MEDIUM` records should go through additional review, correction, or confidence scoring.

In [72]:
# Isolate medium-quality records - 2024 to 2026
recent_medium = recent_clean_check[
    recent_clean_check["location_quality"] == "MEDIUM"
].copy()

recent_medium.shape

(9140, 84)

In [73]:
# Check hat types of location errors drive this category
recent_medium["location_error_type"].value_counts()

location_error_type
FAR_FROM_CENTERLINE         3499
INTERSECTING_ROUTE_ERROR    3204
BLOCKKEY_ERROR              2436
OTHER_LOCATION_ERROR           1
Name: count, dtype: int64

In [74]:
# Percentages
recent_medium["location_error_type"].value_counts(normalize=True).mul(100).round(2)

location_error_type
FAR_FROM_CENTERLINE         38.28
INTERSECTING_ROUTE_ERROR    35.05
BLOCKKEY_ERROR              26.65
OTHER_LOCATION_ERROR         0.01
Name: proportion, dtype: float64

In [75]:
# Check if concentrated by ward
recent_medium["WARD"].value_counts(dropna=False)

WARD
WARD 6    1848
WARD 8    1768
WARD 2    1699
WARD 5    1646
WARD 7    1049
WARD 1     405
WARD 4     394
WARD 3     328
NULL         3
Name: count, dtype: Int64

In [76]:
# Compare to all recent records
recent_clean_check["WARD"].value_counts(dropna=False)

WARD
WARD 2    8138
WARD 5    7127
WARD 7    6538
WARD 6    6534
WARD 8    6027
WARD 1    3854
WARD 4    3771
WARD 3    2359
NULL         4
Name: count, dtype: Int64

In [77]:
# Percentages
medium_by_ward = recent_medium["WARD"].value_counts(normalize=True).mul(100).round(2)
all_recent_by_ward = recent_clean_check["WARD"].value_counts(normalize=True).mul(100).round(2)

medium_by_ward, all_recent_by_ward

(WARD
 WARD 6    20.22
 WARD 8    19.34
 WARD 2    18.59
 WARD 5    18.01
 WARD 7    11.48
 WARD 1     4.43
 WARD 4     4.31
 WARD 3     3.59
 NULL       0.03
 Name: proportion, dtype: Float64,
 WARD
 WARD 2    18.35
 WARD 5    16.07
 WARD 7    14.74
 WARD 6    14.73
 WARD 8    13.59
 WARD 1     8.69
 WARD 4      8.5
 WARD 3     5.32
 NULL       0.01
 Name: proportion, dtype: Float64)

In [78]:
# Check whether MEDIUM records are more severe on average
recent_clean_check.groupby("location_quality")[
    ["severity_score", "total_fatalities", "total_major_injuries", "total_minor_injuries", "total_injuries"]
].mean().round(2)

,severity_score,total_fatalities,total_major_injuries,total_minor_injuries,total_injuries
location_quality,,,,,
HIGH,2.01,0.0,0.02,0.31,0.32
MEDIUM,2.09,0.0,0.02,0.31,0.33


In [79]:
# Highest severity MEDIUM records
recent_medium[
    [
        "REPORTDATE",
        "ADDRESS",
        "WARD",
        "LATITUDE",
        "LONGITUDE",
        "location_error_type",
        "total_fatalities",
        "total_major_injuries",
        "total_minor_injuries",
        "total_injuries",
        "severity_score"
    ]
].sort_values("severity_score", ascending=False).head(20)

,REPORTDATE,ADDRESS,WARD,LATITUDE,LONGITUDE,location_error_type,total_fatalities,total_major_injuries,total_minor_injuries,total_injuries,severity_score
329146,2025-04-23 03:30:00+00:00,1601 16TH STREET SE,WARD 8,38.870084,-76.983077,FAR_FROM_CENTERLINE,1.0,1.0,1.0,2.0,113.0
344428,2026-03-02 02:36:00+00:00,5412 COLORADO AVENUE NW,WARD 4,38.955657,-77.033704,INTERSECTING_ROUTE_ERROR,1.0,1.0,0.0,1.0,110.0
305228,2024-02-10 02:40:00+00:00,"EAST CAPITOL STREET SE\nWASHINGTON,",WARD 7,38.889796,-76.964695,INTERSECTING_ROUTE_ERROR,1.0,1.0,0.0,1.0,110.0
343364,2026-02-06 21:54:00+00:00,3072 STANTON ROAD SE,WARD 8,38.853119,-76.980726,FAR_FROM_CENTERLINE,1.0,1.0,0.0,1.0,110.0
333616,2025-07-13 01:06:00+00:00,850 HOWARD ROAD SE,WARD 8,38.864990,-76.995003,FAR_FROM_CENTERLINE,1.0,0.0,2.0,2.0,106.0
303641,2024-01-10 23:30:00+00:00,"OHIO DRIVE SW\nWASHINGTON,",WARD 2,38.878433,-77.038354,INTERSECTING_ROUTE_ERROR,1.0,0.0,1.0,1.0,103.0
342658,2026-01-22 21:00:00+00:00,"INTERSTATE 295 INTERSTATE BN\nWASHINGTON,",WARD 8,38.822745,-77.016594,INTERSECTING_ROUTE_ERROR,1.0,0.0,0.0,0.0,100.0
340148,2025-11-22 22:05:00+00:00,2637 BARRY ROAD SE,WARD 8,38.861442,-77.001998,INTERSECTING_ROUTE_ERROR,1.0,0.0,0.0,0.0,100.0
313731,2024-07-04 20:24:00+00:00,850 WHARF STREET SW,WARD 6,38.879270,-77.025264,INTERSECTING_ROUTE_ERROR,1.0,0.0,0.0,0.0,100.0
308464,2024-04-10 03:52:00+00:00,2200 SOUTH CAPITOL STREET SE,WARD 8,38.865143,-77.002561,BLOCKKEY_ERROR,1.0,0.0,0.0,0.0,100.0


In [81]:
# Center-line error records
recent_centerline = recent_clean_check[
    recent_clean_check["location_error_type"] == "FAR_FROM_CENTERLINE"
].copy()

recent_centerline[
    [
        "REPORTDATE",
        "ADDRESS",
        "WARD",
        "LATITUDE",
        "LONGITUDE",
        "MAR_ADDRESS",
        "MAR_SCORE",
        "location_error_type",
        "severity_score"
    ]
].head(20)

,REPORTDATE,ADDRESS,WARD,LATITUDE,LONGITUDE,MAR_ADDRESS,MAR_SCORE,location_error_type,severity_score
303143,2024-01-01 02:30:00+00:00,601 4TH PLACE SW,WARD 6,38.882051,-77.018852,601 4TH PLACE SW,200.0,FAR_FROM_CENTERLINE,1.0
303228,2024-01-02 23:40:00+00:00,4103 BENNING ROAD NE,WARD 7,38.892686,-76.943320,4103 BENNING ROAD NE,200.0,FAR_FROM_CENTERLINE,3.0
303249,2024-01-03 10:00:00+00:00,500 INDIANA AVENUE NW,WARD 6,38.894085,-77.018905,500 INDIANA AVENUE NW,200.0,FAR_FROM_CENTERLINE,10.0
303269,2024-01-03 21:40:00+00:00,6000-BLK KANSAS AVE NW,WARD 4,38.963727,-77.009694,KANSAS AVENUE NW FROM PEABODY STREET NW TO KAN...,100.0,FAR_FROM_CENTERLINE,1.0
303294,2024-01-03 22:50:00+00:00,4660 MARTIN LUTHER KING JR AVENUE SW,WARD 8,38.822472,-77.010474,4660 MARTIN LUTHER KING JR AVENUE SW,200.0,FAR_FROM_CENTERLINE,6.0
303315,2024-01-04 22:30:00+00:00,401 MICHIGAN AVENUE NE,WARD 5,38.931090,-77.000049,401 MICHIGAN AVENUE NE,200.0,FAR_FROM_CENTERLINE,3.0
303333,2024-01-05 02:05:00+00:00,22 MICHIGAN AVENUE NW,WARD 5,38.925951,-77.010179,22 MICHIGAN AVENUE NW,200.0,FAR_FROM_CENTERLINE,1.0
303345,2024-01-05 16:33:00+00:00,400 BROOKLAND GROVE DRIVE NE,WARD 5,38.928881,-77.000186,400 BROOKLAND GROVE DRIVE NE,200.0,FAR_FROM_CENTERLINE,3.0
303396,2024-01-06 08:17:00+00:00,500 INDIANA AVENUE NW,WARD 6,38.894085,-77.018905,500 INDIANA AVENUE NW,200.0,FAR_FROM_CENTERLINE,1.0
303443,2024-01-07 03:40:00+00:00,2700 NEW YORK AVENUE NE,WARD 5,38.917977,-76.970600,2700 NEW YORK AVENUE NE,200.0,FAR_FROM_CENTERLINE,10.0


## Medium Location Quality Inspection

The `MEDIUM` location-quality category contains 9,140 recent records from 2024–2026. These records are mostly driven by three specific location-error types: crashes too far from the centerline, intersecting route errors, and blockkey errors.

Among recent `MEDIUM` records, 38.28% are `FAR_FROM_CENTERLINE`, 35.05% are `INTERSECTING_ROUTE_ERROR`, and 26.65% are `BLOCKKEY_ERROR`. This means the medium-quality category mostly captures concrete road-network matching problems, not generic missing data.

The ward comparison suggests that medium-quality records are especially concentrated in Wards 6, 8, 2, and 5. This may reflect differences in roadway geometry, intersection complexity, or network-matching performance across the city, though this would need more investigation before making a causal claim.

The severity comparison is important: `MEDIUM` records have nearly the same average severity score and average injury count as `HIGH` records. This means medium-quality records should not be excluded from the analysis. Some fatal and serious-injury crashes are classified as `MEDIUM`, so dropping them would remove important safety events.

IMPORTANT CLEANING DECISION: keep medium-quality records in the working dataset, but preserve the `location_quality` and `location_error_type` fields so the dashboard can flag records that may be less reliable for precise corridor, road-segment, or intersection-level analysis.


In [82]:
# Create the actual recent working dataset
# Our scope - 2024 to 2026 + everything done above
recent_clean = clean[
    (clean["REPORTDATE"] >= "2024-01-01") &
    (clean["REPORTDATE"] < "2027-01-01")
].copy()

recent_clean.shape

(44352, 84)

In [94]:
# Saves Historical Data Set
clean.to_csv("../data/processed/crashes_clean_full.csv", index=False)

# Saves 2024-2026 Data Set
recent_clean.to_csv("../data/processed/crashes_clean_recent.csv", index=False)

In [84]:
# Check
import os

os.listdir("../data/processed")

['crashes_clean_recent.csv', 'crashes_clean_full.csv']

## Cleaned Data Export

I saved two cleaned datasets:

1. `crashes_clean_full.csv`: the full historical crash dataset with cleaned and derived fields.
2. `crashes_clean_recent.csv`: the 2024–2026 subset that will serve as the main working dataset for the prototype.

The recent dataset is the main focus.

# Validation Pipeline

My initial validation plan was to predict `NEARESTINTKEY` by using other high-quality crash records as the reference pool. The idea was to take `HIGH` location-quality records with known `NEARESTINTKEY` values, hide the target value for a validation sample, and then predict it by finding the nearest similar crash record.

After finding the official `Intersection_Points.csv` dataset, I changed the validation approach. Instead of using nearby crash records as the reference layer, I now use official intersection points. This is cleaner and more defensible because `NEARESTINTKEY` should correspond to an official intersection/network identifier, not simply to the nearest previous crash.

The validation pipeline now works as follows:

1. Use the cleaned 2024–2026 crash dataset as the working crash dataset.
2. Use `NEARESTINTKEY` as the first target because it is nearly complete in recent crash records, unlike `STREETSEGID` and `ROADWAYSEGID`.
3. Build a validation sample from `HIGH` location-quality crash records where `NEARESTINTKEY` is already known.
4. Stratify the validation sample by severity so the test set includes enough fatal, major-injury, minor/unknown-injury, and property-damage-only crashes.
5. Save two validation files:
   - `validation_truth.csv`, which keeps the original `NEARESTINTKEY` as the answer key.
   - `validation_inputs.csv`, which removes `NEARESTINTKEY` so the snapping method has to recreate it.
6. Load the official intersection point dataset.
7. For each validation crash, find the nearest official intersection point using latitude and longitude.
8. Assign that intersection point’s `INTERSECTIONKEY` as the predicted `NEARESTINTKEY`.
9. Compare the predicted `NEARESTINTKEY` against the true `NEARESTINTKEY` from the validation truth file.
10. Evaluate accuracy overall, by severity bucket, and by distance to the matched intersection.

In [165]:
# Looking through my cleaned data and the newly uploaded intersections data
# The date parsing lines make sure REPORTDATE and FROMDATE are treated as actual datetime columns again.
# The final line checks the sizes of both datasets so we know they loaded correctly.
import pandas as pd
import numpy as np

# Load cleaned recent crash data
recent_clean = pd.read_csv("../data/processed/crashes_clean_recent.csv", low_memory=False)

# Load official intersection point data
intersections = pd.read_csv("../data/raw/Intersection_Points.csv", low_memory=False)

# Re-parse date columns after loading from CSV
recent_clean["REPORTDATE"] = pd.to_datetime(recent_clean["REPORTDATE"], errors="coerce")
recent_clean["FROMDATE"] = pd.to_datetime(recent_clean["FROMDATE"], errors="coerce")

recent_clean.shape, intersections.shape

((44352, 85), (19090, 34))

In [166]:
# # Check completeness of possible network ID fields in the crash data
# Reminder to use NEARESTINTKEY
network_id_cols = [
    "STREETSEGID",
    "ROADWAYSEGID",
    "BLOCKKEY",
    "CORRIDORID",
    "NEARESTINTKEY",
    "NEARESTINTROUTEID"
]

recent_clean[network_id_cols].isna().mean().mul(100).round(2)

STREETSEGID          99.86
ROADWAYSEGID         99.86
BLOCKKEY              0.02
CORRIDORID            7.38
NEARESTINTKEY         0.04
NEARESTINTROUTEID     0.04
dtype: float64

In [168]:
# Inspect intersection columns
# Confirm we have: LATITUDE, LONGITUDE, INTERSECTIONKEY, NAME
intersections.columns.tolist()

['X',
 'Y',
 'OBJECTID',
 'MAR_ID',
 'NAME',
 'X_COORDINATE',
 'Y_COORDINATE',
 'LATITUDE',
 'LONGITUDE',
 'WARD',
 'STATUS',
 'ROUTEID_1',
 'ROUTEID_2',
 'STREET_1_NAME',
 'STREET_1_TYPE',
 'STREET_1_QUADRANT',
 'STREET_1_FULL',
 'STREET_2_NAME',
 'STREET_2_TYPE',
 'STREET_2_QUADRANT',
 'STREET_2_FULL',
 'INTERSECTIONKEY',
 'INTERSECTION_TYPE',
 'NATIONAL_GRID',
 'SOURCE',
 'BEFORE_DATE',
 'BEFORE_DATE_SOURCE',
 'BEGIN_DATE',
 'BEGIN_DATE_SOURCE',
 'FIRST_KNOWN_DATE',
 'FIRST_KNOWN_DATE_SOURCE',
 'CREATED_DATE',
 'LAST_EDITED_DATE',
 'METADATA_ID']

In [171]:
# Cleaning intersection_points

# Keep only intersections with coordinates and an intersection key
intersections_clean = intersections[
    intersections["LATITUDE"].notna() &
    intersections["LONGITUDE"].notna() &
    intersections["INTERSECTIONKEY"].notna()
].copy()

# Convert coordinates to numeric
intersections_clean["LATITUDE"] = pd.to_numeric(intersections_clean["LATITUDE"], errors="coerce")
intersections_clean["LONGITUDE"] = pd.to_numeric(intersections_clean["LONGITUDE"], errors="coerce")

# Keep only active intersections if STATUS is available
intersections_clean = intersections_clean[
    intersections_clean["STATUS"].astype(str).str.upper() == "ACTIVE"
].copy()

# Drop duplicate intersection points
intersections_clean = intersections_clean.drop_duplicates(
    subset=["INTERSECTIONKEY", "LATITUDE", "LONGITUDE"]
).copy()

intersections_clean.shape

(8424, 34)

## Cleaned Intersection Reference Layer

After filtering the intersection point dataset, I kept 8,424 active intersections with valid coordinates and an `INTERSECTIONKEY`.

This cleaned intersection layer will serve as the official reference for the recovery baseline. For each validation crash, I will find the nearest official intersection point and use that point’s `INTERSECTIONKEY` as the predicted `NEARESTINTKEY`.

In [172]:
target_col = "NEARESTINTKEY"

source_pool = recent_clean[
    (recent_clean["location_quality"] == "HIGH") &
    (recent_clean["has_valid_lat_lon"] == True) &
    (recent_clean[target_col].notna())
].copy()

source_pool.shape

(35212, 85)

In [173]:
def assign_eval_severity(row):
    if row["total_fatalities"] > 0:
        return "fatal"
    elif row["total_major_injuries"] > 0:
        return "major"
    elif row["total_minor_injuries"] > 0 or row["total_unknown_injuries"] > 0:
        return "minor"
    else:
        return "pdo"

source_pool["severity_bucket"] = source_pool.apply(assign_eval_severity, axis=1)

source_pool["severity_bucket"].value_counts()

severity_bucket
pdo      26318
minor     8293
major      538
fatal       63
Name: count, dtype: int64

## Trusted Source Pool Check

The trusted source pool contains 35,212 recent crash records. These records are `HIGH` location quality, have valid-looking latitude/longitude, and have a known `NEARESTINTKEY`.

The source pool includes enough records in each severity bucket to create the planned 500-record validation sample: 30 fatal crashes, 80 major-injury crashes, 200 minor/unknown-injury crashes, and 190 property-damage-only crashes.

This confirms that the severity-stratified validation design is feasible.

In [175]:
# Create the 500-row validation sample

target_per_bucket = {
    "fatal": 30,
    "major": 80,
    "minor": 200,
    "pdo": 190
}

sample_parts = []

for bucket, target_n in target_per_bucket.items():
    bucket_df = source_pool[source_pool["severity_bucket"] == bucket]
    actual_n = min(target_n, len(bucket_df))

    print(f"{bucket}: sampling {actual_n} of {len(bucket_df)} available")

    sample_parts.append(
        bucket_df.sample(n=actual_n, random_state=42)
    )

validation_sample = pd.concat(sample_parts).sample(frac=1, random_state=42).copy()

validation_sample.shape

fatal: sampling 30 of 63 available
major: sampling 80 of 538 available
minor: sampling 200 of 8293 available
pdo: sampling 190 of 26318 available


(500, 86)

## Validation Sample Created

The severity-stratified validation sample was created successfully. The final sample contains 500 crash records: 30 fatal crashes, 80 major-injury crashes, 200 minor/unknown-injury crashes, and 190 property-damage-only crashes.

In [177]:
# Save selected validation IDs so this exact split can be reproduced later
# This file is essentially the record of the 500 crashes we decided to choose
validation_indices = validation_sample[
    [
        "CRIMEID",
        "severity_bucket"
    ]
].copy()

validation_indices.to_csv(
    "../data/processed/validation_indices.csv",
    index=False
)

In [178]:
# Create the validation truth file
# This KEEPS the real NEARESTINTKEY
truth_cols = [
    "CRIMEID",
    "CCN",
    "REPORTDATE",
    "ADDRESS",
    "WARD",
    "LATITUDE",
    "LONGITUDE",
    "severity_bucket",
    "NEARESTINTKEY"
]

validation_truth = validation_sample[truth_cols].copy()

validation_truth = validation_truth.rename(columns={
    "NEARESTINTKEY": "true_nearestintkey"
})

validation_truth.head()

,CRIMEID,CCN,REPORTDATE,ADDRESS,WARD,LATITUDE,LONGITUDE,severity_bucket,true_nearestintkey
32997,66026433919,25138936,2025-09-11 15:30:00+00:00,5002 HAYES STREET NE,WARD 7,38.900023,-76.930118,pdo,eec3f8ec116089b5f9f3e113946742b6
18884,61411145262,24188626,2024-12-06 00:30:00+00:00,1200 MASSACHUSETTS AVENUE NW,WARD 2,38.904114,-77.028083,major,70df05896c27dbbf205a31da1937915d
20059,62040455653,24200720,2024-12-28 05:00:00+00:00,2000 38TH STREET SE,WARD 7,38.865072,-76.952216,pdo,f3a7c7a0ff5ef2117753ea8b1d0dd199
12511,58446682916,24126534,2024-08-18 03:19:00+00:00,1702 25TH STREET SE,WARD 7,38.869472,-76.970708,minor,5d1b5282a5985091042467ba32f0c0ca
6080,55678473950,24061726,2024-04-24 15:00:00+00:00,2900 M STREET NW,WARD 2,38.905092,-77.058105,major,4160c6f3ff10cfd4469cd16f79a77f87


In [179]:
# Create the validation input file
# This REMOVES NEARESTINTKEY so the snapping method has to recreate it
validation_inputs = validation_sample.copy()

validation_inputs["NEARESTINTKEY"] = pd.NA

validation_inputs[
    [
        "CRIMEID",
        "CCN",
        "REPORTDATE",
        "ADDRESS",
        "WARD",
        "LATITUDE",
        "LONGITUDE",
        "severity_bucket",
        "NEARESTINTKEY"
    ]
].head()

,CRIMEID,CCN,REPORTDATE,ADDRESS,WARD,LATITUDE,LONGITUDE,severity_bucket,NEARESTINTKEY
32997,66026433919,25138936,2025-09-11 15:30:00+00:00,5002 HAYES STREET NE,WARD 7,38.900023,-76.930118,pdo,<NA>
18884,61411145262,24188626,2024-12-06 00:30:00+00:00,1200 MASSACHUSETTS AVENUE NW,WARD 2,38.904114,-77.028083,major,<NA>
20059,62040455653,24200720,2024-12-28 05:00:00+00:00,2000 38TH STREET SE,WARD 7,38.865072,-76.952216,pdo,<NA>
12511,58446682916,24126534,2024-08-18 03:19:00+00:00,1702 25TH STREET SE,WARD 7,38.869472,-76.970708,minor,<NA>
6080,55678473950,24061726,2024-04-24 15:00:00+00:00,2900 M STREET NW,WARD 2,38.905092,-77.058105,major,<NA>


In [180]:
# Save both files
validation_truth.to_csv(
    "../data/processed/validation_truth.csv",
    index=False
)

validation_inputs.to_csv(
    "../data/processed/validation_inputs.csv",
    index=False
)

In [181]:
# Quick check
validation_truth.shape, validation_inputs.shape

((500, 9), (500, 86))

In [182]:
validation_inputs["severity_bucket"].value_counts()

severity_bucket
minor    200
pdo      190
major     80
fatal     30
Name: count, dtype: int64

In [219]:
# 1. Distribution of match scores
print("Match score distribution:")
print(top5_candidates["street_match_score"].value_counts().sort_index())

# 2. How many crashes have AT LEAST ONE candidate with score > 0?
crashes_with_any_match = (
    top5_candidates.groupby("CRIMEID")["street_match_score"].max() > 0
).mean()
print(f"\nCrashes with at least one matched candidate: {crashes_with_any_match:.1%}")

# 3. Look at 5 failures and inspect what their address vs candidates look like
failures_v3 = rerank_eval[~rerank_eval["correct"]].head(5)
for crime_id in failures_v3["CRIMEID"]:
    print(f"\n=== {crime_id} ===")
    print(top5_candidates[top5_candidates["CRIMEID"] == crime_id][
        ["candidate_rank", "candidate_street_1", "candidate_street_2",
         "ADDRESS", "street_match_score", "candidate_distance_m"]
    ].to_string())

Match score distribution:
street_match_score
0    1260
1    1215
2      25
Name: count, dtype: int64

Crashes with at least one matched candidate: 97.8%

=== 53853522863 ===
      candidate_rank candidate_street_1 candidate_street_2                 ADDRESS  street_match_score  candidate_distance_m
1005               1  CAROLINA PLACE NW  ARIZONA AVENUE NW  5306 CAROLINA PLACE NW                   1             23.131180
1006               2  ARIZONA AVENUE NW   DORSETT PLACE NW  5306 CAROLINA PLACE NW                   0             52.573957
1007               3  ARIZONA AVENUE NW  POTOMAC AVENUE NW  5306 CAROLINA PLACE NW                   0             90.626719
1008               4  CAROLINA PLACE NW    GALENA PLACE NW  5306 CAROLINA PLACE NW                   1            142.478224
1009               5    GALENA PLACE NW   DORSETT PLACE NW  5306 CAROLINA PLACE NW                   0            151.007908

=== 53872316486 ===
      candidate_rank      candidate_street_1      candi

In [224]:
# Check whether these fields already exist in validation_inputs
nis_cols = [
    "NEARESTINTSTREETNAME",
    "OFFINTERSECTION",
    "INTAPPROACHDIRECTION"
]

[c for c in nis_cols if c in validation_inputs.columns]

['NEARESTINTSTREETNAME', 'OFFINTERSECTION', 'INTAPPROACHDIRECTION']

In [227]:
val_with_nis = validation_inputs.copy()

print("NEARESTINTSTREETNAME populated:", val_with_nis["NEARESTINTSTREETNAME"].notna().mean().round(3))
print("OFFINTERSECTION populated:", val_with_nis["OFFINTERSECTION"].notna().mean().round(3))
print("INTAPPROACHDIRECTION populated:", val_with_nis["INTAPPROACHDIRECTION"].notna().mean().round(3))

val_with_nis[
    [
        "CRIMEID",
        "ADDRESS",
        "NEARESTINTSTREETNAME",
        "OFFINTERSECTION",
        "INTAPPROACHDIRECTION"
    ]
].head(10)

NEARESTINTSTREETNAME populated: 1.0
OFFINTERSECTION populated: 1.0
INTAPPROACHDIRECTION populated: 1.0


,CRIMEID,ADDRESS,NEARESTINTSTREETNAME,OFFINTERSECTION,INTAPPROACHDIRECTION
32997,66026433919,5002 HAYES STREET NE,50TH ST NE,45.93,WEST
18884,61411145262,1200 MASSACHUSETTS AVENUE NW,L ST NW,62.88,NORTHWEST
20059,62040455653,2000 38TH STREET SE,ALABAMA AVE SE,20.42,SOUTH
12511,58446682916,1702 25TH STREET SE,R ST SE,18.74,SOUTH
6080,55678473950,2900 M STREET NW,M ST NW,16.21,SOUTH
36485,67412038186,200 MASSACHUSETTS AVENUE NW,2ND ST NW,19.42,NORTHWEST
43094,69676991093,1624 U STREET NW,U ST NW,26.59,SOUTH
26258,63778252286,751 8TH STREET SE,I ST SE,31.59,NORTH
32303,65594696923,"NEW YORK AVENUE NW\nWASHINGTON,",5TH ST NW,35.03,NORTHEAST
23854,63232557747,4506 BOWEN ROAD SE,46TH ST SE,38.60,WEST


## Baseline v3: Top-5 Re-Ranking with Both Street Signals

The previous re-ranking method used simple address matching, but that only captures one side of an intersection. Many crash addresses identify the block or primary street, while the field `NEARESTINTSTREETNAME` often provides the cross street or nearest intersection street.

Baseline v3 uses both signals. For each crash, I generate the top 5 nearest official intersection candidates. Then I score each candidate based on whether its two official street names appear in either the crash `ADDRESS` or the crash `NEARESTINTSTREETNAME`.

This is a more realistic re-ranking method because the correct intersection is usually defined by two street names, not just distance or one address string.

Logic:

+2 if candidate street appears in ADDRESS

+2 if candidate street appears in NEARESTINTSTREETNAME

+bonus if both candidate streets are matched somewhere

-distance penalty so closer candidates still matter

In [228]:
def normalize_text(value):
    if pd.isna(value):
        return ""

    text = str(value).upper()

    replacements = {
        ".": "",
        ",": "",
        "\n": " ",
        " AVENUE ": " AVE ",
        " STREET ": " ST ",
        " ROAD ": " RD ",
        " DRIVE ": " DR ",
        " PLACE ": " PL ",
        " COURT ": " CT ",
        " CIRCLE ": " CIR ",
        " BOULEVARD ": " BLVD ",
        " NORTHWEST": " NW",
        " NORTHEAST": " NE",
        " SOUTHWEST": " SW",
        " SOUTHEAST": " SE",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return " ".join(text.split())


def both_street_score(row):
    address = normalize_text(row["ADDRESS"])
    nearest_street = normalize_text(row.get("NEARESTINTSTREETNAME", ""))

    street_1 = normalize_text(row["candidate_street_1"])
    street_2 = normalize_text(row["candidate_street_2"])

    score = 0
    matched_streets = 0

    for street in [street_1, street_2]:
        street_matched = False

        if street and street in address:
            score += 2
            street_matched = True

        if street and street in nearest_street:
            score += 2
            street_matched = True

        if street_matched:
            matched_streets += 1

    # Bonus if both streets in the candidate intersection are supported
    if matched_streets == 2:
        score += 3

    return score

In [230]:
# Score and rerank
top5_candidates["both_street_score"] = top5_candidates.apply(both_street_score, axis=1)

top5_candidates["v3_rerank_score"] = (
    top5_candidates["both_street_score"] * 100
    - top5_candidates["candidate_distance_m"]
)

v3_predictions = (
    top5_candidates
    .sort_values(
        by=["CRIMEID", "v3_rerank_score", "candidate_rank"],
        ascending=[True, False, True]
    )
    .groupby("CRIMEID", as_index=False)
    .first()
)

v3_predictions = v3_predictions.rename(columns={
    "candidate_intersectionkey": "predicted_nearestintkey",
    "candidate_name": "matched_intersection_name",
    "candidate_distance_m": "matched_distance_m"
})

In [231]:
# Evaluate
v3_eval = v3_predictions.merge(
    validation_truth[["CRIMEID", "true_nearestintkey"]],
    on="CRIMEID",
    how="left"
)

v3_eval["correct"] = (
    v3_eval["predicted_nearestintkey"].astype(str)
    == v3_eval["true_nearestintkey"].astype(str)
)

v3_eval["correct"].mean()

np.float64(0.904)

## Baseline v3 Result: Top-5 Re-Ranking with Both Street Signals

Baseline v3 improves the snapping method by using both spatial proximity and street-name evidence. Instead of automatically choosing the nearest official intersection, the method first generates the top 5 nearest official intersection candidates. It then re-ranks those candidates using both the crash `ADDRESS` and `NEARESTINTSTREETNAME`.

This approach correctly recovered the hidden `NEARESTINTKEY` for 90.4% of validation records. This is a major improvement over the distance-only official-intersection baseline, which recovered 73.0%.

The result suggests that most crashes can be matched to the correct official intersection when the pipeline combines geography with street-name context. Distance alone often gets the right area, but street-name evidence is needed to choose the correct intersection among nearby candidates.

In [233]:
v3_eval["correct"].value_counts(normalize=True).mul(100).round(2)

correct
True     90.4
False     9.6
Name: proportion, dtype: float64

In [234]:
v3_severity_results = (
    v3_eval
    .groupby("severity_bucket")
    .agg(
        n_records=("correct", "size"),
        accuracy=("correct", "mean"),
        median_distance_m=("matched_distance_m", "median")
    )
)

v3_severity_results["accuracy"] = v3_severity_results["accuracy"].mul(100).round(2)
v3_severity_results["median_distance_m"] = v3_severity_results["median_distance_m"].round(2)

v3_severity_results

,n_records,accuracy,median_distance_m
severity_bucket,,,
fatal,30,96.67,37.08
major,80,87.50,31.93
minor,200,92.50,31.23
pdo,190,88.42,45.44


In [238]:
results_table = pd.DataFrame([
    {
        "method": "V1: nearest official intersection",
        "overall_accuracy": 73.0
    },
    {
        "method": "V2: top-5 + address matching",
        "overall_accuracy": 79.8
    },
    {
        "method": "V3: top-5 + both-street scoring",
        "overall_accuracy": 90.4
    }
])

results_table

,method,overall_accuracy
0,V1: nearest official intersection,73.0
1,V2: top-5 + address matching,79.8
2,V3: top-5 + both-street scoring,90.4
